# 10,000 半透明ブロック敷き詰め (grid_env_10k)

30 m 床の上に **100×100 = 10,000** 個の `BP_TransparentCube`（`SetBlocking False`）を敷き詰め、Humanoid / SpotDog を `grid_env_hri` と同じ位置に配置します。

ロジック: `grid_env_10k.py`（スポーン実装は `grid_env_hri_simulation` を再利用）

## ブロック座標（1 始まり）

- 床 **左下角** のマス = **(1, 1)**（マップ連続座標の (0,0) m 付近）
- Humanoid から見て **右**: (1,1), (2,1), …, (100,1) → **gx** が増える
- Humanoid から見て **正面**: (1,1), (1,2), …, (1,100) → **gy** が増える
- Actor 名: `block_{gx:03d}_{gy:03d}`（例: `block_050_050`）

## 前提

1. Windows: `SimWorld.exe -windowed -log /Game/Maps/empty.umap`
2. pakchunk9002（床・TransparentCube）
3. カーネル: `conda activate simworld`
4. 10,000 個スポーンは **20〜40 分** 程度（`BLOCK_SPAWN_INTERVAL_S` 既定 0.01）

接続前に **`importlib.reload(g10k)`** と **`importlib.reload(geh)`** を推奨。

In [ ]:
import importlib
import os
import sys
from pathlib import Path
from typing import Optional, Tuple


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_g10k_dir = _root / "dev" / "grid_env_10k"
_geh_dir = _root / "dev" / "grid_env_hri"
for p in (_root, _g10k_dir, _geh_dir):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from simworld.communicator.communicator import Communicator
from simworld.communicator.unrealcv import UnrealCV

ucv: Optional[UnrealCV] = None
communicator: Optional[Communicator] = None
block_registry: dict = {}
human_name: Optional[str] = None
robot_ok: bool = False

print(f"[Paths] root={_root}")
print(f"[Paths] grid_env_10k={_g10k_dir}")

In [ ]:
import grid_env_10k as g10k
import grid_env_hri_simulation as geh


def ensure_connection() -> Tuple[UnrealCV, Communicator]:
    global ucv, communicator
    if ucv is not None and ucv.client.isconnected():
        if communicator is None or communicator.unrealcv is not ucv:
            communicator = Communicator(ucv)
        return ucv, communicator
    ucv, communicator = g10k.ensure_connection()
    return ucv, communicator

In [ ]:
ucv, communicator = ensure_connection()

In [ ]:
# 本番: BLOCK_GRID_N = 100（10,000 個） / 試験: 5（25 個）
BLOCK_GRID_N = 100
BLOCK_SPAWN_INTERVAL_S = 0.01

os.environ["BLOCK_GRID_N"] = str(BLOCK_GRID_N)
os.environ["BLOCK_SPAWN_INTERVAL_S"] = str(BLOCK_SPAWN_INTERVAL_S)

importlib.reload(geh)
importlib.reload(g10k)

g10k.configure_spawn_env(
    grid_n=BLOCK_GRID_N,
    spawn_interval_s=BLOCK_SPAWN_INTERVAL_S,
)

print(
    f"[Config] {BLOCK_GRID_N}×{BLOCK_GRID_N} = {BLOCK_GRID_N**2} blocks, "
    f"interval={BLOCK_SPAWN_INTERVAL_S}s, "
    f"human={geh.HUMAN_MAP_XY_M} m, robot={geh.ROBOT_MAP_XY_M} m"
)

In [ ]:
floor_ok, ucv = g10k.spawn_floor_with_retry(ucv)
if not floor_ok:
    raise RuntimeError("[Floor] spawn failed")

In [ ]:
g10k.prepare_ue_session(ucv)

block_registry = g10k.spawn_translucent_block_grid(
    ucv,
    grid_n=BLOCK_GRID_N,
    spawn_interval_s=BLOCK_SPAWN_INTERVAL_S,
)
expected = BLOCK_GRID_N ** 2
if len(block_registry) != expected:
    raise RuntimeError(
        f"[Blocks] incomplete: {len(block_registry)}/{expected} — "
        "interval を増やすか SimWorld を再起動"
    )

In [ ]:
human_name, robot_ok = g10k.spawn_agents(communicator, ucv)
geh.report_spawn_state(ucv, {}, human_name)
g10k.verify_block_samples(ucv, block_registry, grid_n=BLOCK_GRID_N)

## クリーンアップ（任意）

10,000 Actor の削除にも数分かかります。

In [ ]:
RUN_CLEANUP = False

if RUN_CLEANUP:
    g10k.cleanup_all(ucv, block_registry, human_name)
else:
    print("[Cleanup] skipped")